# Laboratório 03 — Busca informada: heurísticas e A\*

**Fundamentos de Inteligência Artificial** · 2026.2 · Prof. Leonardo Garcia Marques

---

Na aula passada, todos os algoritmos eram **o mesmo laço**, mudando só de que ponto
da fronteira se retirava o nó. Hoje isso fica literal: você vai escrever **uma**
função e obter **três** algoritmos dela.

| Estratégia | Prioridade | |
|---|---|---|
| Custo uniforme | `f = g` | só o passado |
| Busca gulosa | `f = h` | só o futuro |
| **A\*** | `f = g + h` | **os dois** |

> ⚙️ Só `collections` e `heapq`, ambos da biblioteca padrão. Continua sem
> dependência externa.


## Parte 0 — O labirinto

Maior que o da aula passada, e com desvios obrigatórios: o caminho ótimo tem **30
passos**, enquanto a distância de Manhattan entre `S` e `G` é apenas **26**.

Rode e veja o mapa.


In [ ]:
from collections import deque
import heapq

LABIRINTO = [
    "S.#.........#.",
    ".....####...#.",
    ".#......##...#",
    "...#....#.....",
    ".#.###..##....",
    "...##..##..#..",
    "##..#####.#.##",
    ".####.#...#...",
    "####.#.#......",
    "##..###....##.",
    "#........###..",
    "..#.###.......",
    ".#....#.......",
    "..........#.#G",
]

LINHAS, COLUNAS = len(LABIRINTO), len(LABIRINTO[0])

def acha(simbolo):
    for i, linha in enumerate(LABIRINTO):
        j = linha.find(simbolo)
        if j >= 0:
            return (i, j)

INICIO   = acha('S')
OBJETIVO = acha('G')

def livre(i, j):
    return 0 <= i < LINHAS and 0 <= j < COLUNAS and LABIRINTO[i][j] != '#'

def vizinhos(estado):
    i, j = estado
    return [(i+di, j+dj) for di, dj in [(-1,0),(1,0),(0,-1),(0,1)] if livre(i+di, j+dj)]

def reconstroi(veio_de, estado):
    caminho = []
    while estado is not None:
        caminho.append(estado)
        estado = veio_de[estado]
    return caminho[::-1]

def desenha(caminho=None):
    marcas = set(caminho or [])
    for i, linha in enumerate(LABIRINTO):
        print(''.join('o' if (i,j) in marcas and c not in 'SG' else c
                      for j, c in enumerate(linha)))

print('Início:', INICIO, ' Objetivo:', OBJETIVO, ' Labirinto:', LINHAS, 'x', COLUNAS)
desenha()


## Parte 1 — A busca cega, para comparar

Esta é a função da aula passada, já pronta. Rode para ter os números de referência.


In [ ]:
def busca_cega(estrategia):
    """Da aula passada. 'bfs' ou 'dfs'. Devolve (caminho, nós expandidos)."""
    fronteira = deque([INICIO])
    veio_de   = {INICIO: None}
    visitados = set()
    expandidos = 0
    while fronteira:
        estado = fronteira.popleft() if estrategia == 'bfs' else fronteira.pop()
        if estado in visitados:
            continue
        visitados.add(estado)
        expandidos += 1
        if estado == OBJETIVO:
            return reconstroi(veio_de, estado), expandidos
        for v in vizinhos(estado):
            if v not in visitados:
                veio_de[v] = estado
                fronteira.append(v)
    return None, expandidos

for e in ['bfs', 'dfs']:
    cam, exp = busca_cega(e)
    print(f'{e.upper():4s} -> {len(cam):3d} células | {exp:3d} expandidos')


## Parte 2 — A heurística

A **distância de Manhattan** é a distância andando só na horizontal e na vertical,
**ignorando as paredes**. É admissível justamente por isso: paredes só podem
aumentar o caminho real, nunca diminuí-lo.


In [ ]:
def h(estado):
    """Distância de Manhattan de `estado` até OBJETIVO."""
    # TODO 1: some as diferenças absolutas de linha e de coluna.
    #         Dica: abs(a - b)
    return 0


In [ ]:
print('h(início) =', h(INICIO), ' | h(objetivo) =', h(OBJETIVO))
assert h(OBJETIVO) == 0, 'a heurística tem de valer 0 no objetivo'
assert h(INICIO) == 26, 'confira as diferenças absolutas'
print('ok')


## Parte 3 — Uma função, três algoritmos

Repare na tupla que vai para a fila: `(f, g, contador, estado)`.

O `heapq` compara elemento a elemento. O **contador** existe para desempatar
quando dois nós têm o mesmo `f` — sem ele, o Python passaria a comparar o campo
seguinte, o que dá errado assim que o estado deixar de ser uma tupla de números.

> ⚠️ O teste de objetivo é feito **ao retirar** o nó da fronteira, nunca ao gerar.
> Testar na geração devolve o primeiro caminho encontrado — que pode não ser o
> mais barato. É um erro silencioso: só aparece quando você compara com a BFS.


In [ ]:
def busca_informada(estrategia):
    """estrategia: 'ucs', 'gulosa' ou 'astar'. Devolve (caminho, expandidos)."""
    contador = 0                       # desempata prioridades iguais
    fronteira = [(0, 0, contador, INICIO)]     # (f, g, contador, estado)
    veio_de   = {INICIO: None}
    melhor_g  = {INICIO: 0}
    visitados = set()
    expandidos = 0

    while fronteira:
        f_atual, g, _, estado = heapq.heappop(fronteira)
        if estado in visitados:
            continue
        visitados.add(estado)
        expandidos += 1

        # TODO 2: teste de objetivo — AO RETIRAR, nunca ao gerar.
        #         Devolva  reconstroi(veio_de, estado), expandidos

        for v in vizinhos(estado):
            novo_g = g + 1
            if v in melhor_g and novo_g >= melhor_g[v]:
                continue                      # já conhecemos caminho melhor
            melhor_g[v] = novo_g
            veio_de[v]  = estado
            contador += 1

            # TODO 3: calcule a prioridade f conforme a estratégia:
            #     'ucs'    -> f = novo_g          (só o passado)
            #     'gulosa' -> f = h(v)            (só o futuro)
            #     'astar'  -> f = novo_g + h(v)   (os dois)
            f = 0   # <-- substitua

            heapq.heappush(fronteira, (f, novo_g, contador, v))

    return None, expandidos


## Parte 4 — A tabela

Rode e compare. Duas colunas contam a história: quantos nós cada estratégia
expandiu, e se o caminho encontrado é o ótimo.


In [ ]:
otimo = len(busca_cega('bfs')[0])

print(f"{'algoritmo':<18}{'caminho':>9}{'expandidos':>12}{'ótimo?':>10}")
print('-' * 49)

for nome, fn, arg in [('BFS', busca_cega, 'bfs'),
                      ('DFS', busca_cega, 'dfs'),
                      ('Custo uniforme', busca_informada, 'ucs'),
                      ('Busca gulosa', busca_informada, 'gulosa'),
                      ('A*', busca_informada, 'astar')]:
    caminho, expandidos = fn(arg)
    marca = 'sim' if len(caminho) == otimo else 'NÃO'
    print(f'{nome:<18}{len(caminho):>9}{expandidos:>12}{marca:>10}')


In [ ]:
# Os dois caminhos lado a lado: o do A* e o da gulosa
for e in ['astar', 'gulosa']:
    caminho, expandidos = busca_informada(e)
    print(f'{e} -> {len(caminho)} células, {expandidos} expandidos')
    desenha(caminho)
    print()


## Parte 5 — Responda no próprio notebook

1. O custo uniforme e a BFS deram exatamente o mesmo resultado. Isso era esperado?
   Em que situação eles passariam a divergir?
2. A busca gulosa expandiu **menos** nós que todas as outras. Por que, ainda assim,
   você não a usaria num GPS?
3. O A\* expandiu menos que a BFS e achou o mesmo caminho. De onde vem essa
   economia?
4. Troque a heurística por `h(estado) * 3` e rode a tabela de novo. O caminho do
   A\* mudou? Ele ainda é ótimo? Explique usando a palavra **admissível**.
5. Troque a heurística por `0` e rode. Em que algoritmo o A\* se transformou?


In [ ]:
# Espaço para os experimentos das questões 4 e 5


---

## Desafio (opcional)

A distância de Manhattan ignora as paredes. Invente uma heurística **mais
informada** que ela — isto é, que dê valores maiores, sem nunca superestimar o
custo real — e meça quantos nós o A\* passa a expandir.

Se conseguir, você terá demonstrado na prática que, entre duas heurísticas
admissíveis, a maior é a melhor.


In [ ]:
# Desafio
